# Mu-SHROOM (Finnish): XLM-RoBERTa-Large token classification

This notebook is a Colab-ready pipeline for Finnish hallucination span detection on `Helsinki-NLP/mu-shroom`.

It covers:
- loading and inspecting the Finnish data,
- converting character spans to token BIO labels with XLM-R offset mappings,
- evaluating an English-finetuned checkpoint on Finnish,
- fine-tuning `xlm-roberta-large` on Finnish,
- evaluating the Finnish-finetuned checkpoint on Finnish,
- exporting predictions in the official Mu-SHROOM format,
- running the official scorer and generating a comparison table.

Recommended Colab runtime: `GPU` with high-RAM if available. `xlm-roberta-large` is memory-heavy, so the default training setup uses mixed precision and gradient accumulation.

In [1]:
# Colab setup
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn pandas

import os
import sys
import json
import math
import inspect
import random
import subprocess
import importlib.util
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import precision_recall_fscore_support, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
import evaluate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQEVAL = evaluate.load("seqeval")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
# Experiment configuration
MODEL_NAME = "xlm-roberta-large"
LANGUAGE = "fi"
SEED = 4650
MAX_LENGTH = 512

# TODO: point this at your English-finetuned checkpoint before running the English-on-Finnish evaluation.
ENGLISH_CHECKPOINT = "/content/drive/MyDrive/xlmr-mu-shroom-en-best"

# TODO: change these paths if you want outputs somewhere else on Colab/Drive.
OUTPUT_ROOT = "/content/mu-shroom-fi-xlmr-large"
FINNISH_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "best")
PREDICTIONS_DIR = os.path.join(OUTPUT_ROOT, "predictions")
SCORER_REPO_DIR = "/content/mu-shroom"

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2
PREDICTION_THRESHOLD = 0.50

label_list = ["O", "B-HALL", "I-HALL"]
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer fast:", tokenizer.is_fast)
print("Output root:", OUTPUT_ROOT)
print("English checkpoint:", ENGLISH_CHECKPOINT)

Tokenizer fast: True
Output root: /content/mu-shroom-fi-xlmr-large
English checkpoint: /content/drive/MyDrive/xlmr-mu-shroom-en-best


In [3]:
import os; print(ENGLISH_CHECKPOINT, os.path.isdir(ENGLISH_CHECKPOINT))

/content/drive/MyDrive/xlmr-mu-shroom-en-best True


## Load Finnish Mu-SHROOM Dataset

This section loads the Finnish subset and prints enough structure to confirm which columns contain:
- the generated answer text,
- the hard hallucination spans,
- the soft per-span probabilities used by the official scorer.

The notebook uses schema inference so it still works if the dataset config exposes the same fields through slightly different split layouts.

In [4]:
def load_mu_shroom_language(language: str) -> DatasetDict:
    try:
        ds = load_dataset("Helsinki-NLP/mu-shroom", language)
        print(f"Loaded direct config: {language}")
        return ds
    except Exception as direct_error:
        print(f"Direct config load failed for {language}: {direct_error}")
        ds = load_dataset("Helsinki-NLP/mu-shroom", "all")
        for candidate_column in ["language", "lang", "locale"]:
            if candidate_column in ds[next(iter(ds.keys()))].column_names:
                filtered = DatasetDict(
                    {
                        split_name: split_ds.filter(lambda row: row[candidate_column] == language)
                        for split_name, split_ds in ds.items()
                    }
                )
                print(f"Loaded 'all' config and filtered on column '{candidate_column}'.")
                return filtered
        raise RuntimeError("Could not locate a language column while filtering the Mu-SHROOM 'all' config.")


def infer_schema(example_row: Dict) -> Dict[str, Optional[str]]:
    columns = list(example_row.keys())
    text_candidates = [
        "model_output_text",
        "answer",
        "output_text",
        "text",
        "response",
    ]
    hard_candidates = ["hard_labels", "hard_spans", "hallucination_spans"]
    soft_candidates = ["soft_labels", "soft_spans", "probabilistic_labels"]
    id_candidates = ["id", "sample_id", "instance_id"]

    def choose(candidates):
        for candidate in candidates:
            if candidate in columns:
                return candidate
        return None

    schema = {
        "text_col": choose(text_candidates),
        "hard_col": choose(hard_candidates),
        "soft_col": choose(soft_candidates),
        "id_col": choose(id_candidates),
    }
    return schema


raw_ds = load_mu_shroom_language(LANGUAGE)
print(raw_ds)
print("Available splits:", list(raw_ds.keys()))
for split_name, split_ds in raw_ds.items():
    print(f"{split_name}: {len(split_ds)} rows")

preview_split_name = "train" if "train" in raw_ds else next(iter(raw_ds.keys()))
preview_row = raw_ds[preview_split_name][0]
schema = infer_schema(preview_row)

print("\nDetected schema:")
for key, value in schema.items():
    print(f"  {key}: {value}")

print("\nColumns in preview split:")
print(raw_ds[preview_split_name].column_names)

print("\nPreview row:")
for key, value in preview_row.items():
    if isinstance(value, str):
        print(f"\n{key}: {value[:350]!r}")
    else:
        print(f"\n{key}: {value}")

if schema["text_col"] is None or schema["hard_col"] is None:
    raise ValueError("Could not infer the answer text column or hard span annotation column. Inspect the printed preview row and update infer_schema().")

Loaded direct config: fi
DatasetDict({
    validation: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_tokens', 'model_output_logits', 'wikipedia_url', 'annotations'],
        num_rows: 50
    })
    test: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_tokens', 'model_output_logits', 'wikipedia_url', 'annotations'],
        num_rows: 150
    })
})
Available splits: ['validation', 'test']
validation: 50 rows
test: 150 rows

Detected schema:
  text_col: model_output_text
  hard_col: hard_labels
  soft_col: soft_labels
  id_col: id

Columns in preview split:
['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_tokens', 'model_output_logits', 'wikipedia_url', 'annotations']

Preview row:

id: 'val-fi-1'

lang: 'FI'

model_input: 'Mistä maakunnista Galicia koostuu?'

## Convert Character Spans to Token Labels

The most error-prone step is label alignment. XLM-R uses subword tokenization, so one character span can cover part of a word, a full word, or several subword tokens. The helper code below:
- normalizes span annotations into `[start, end)` pairs,
- uses offset mappings from the fast tokenizer,
- assigns `B-HALL` to the first overlapping token and `I-HALL` to later overlapping tokens,
- skips special tokens with offset `(0, 0)`,
- preserves no-span examples as all-`O` labels,
- merges token predictions back into character spans for official scoring.

In [5]:
def json_default(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type not serializable: {type(obj)}")


def normalize_hard_spans(raw_spans) -> List[List[int]]:
    if raw_spans is None:
        return []

    normalized = []
    for span in raw_spans:
        if span is None:
            continue
        if isinstance(span, dict):
            start = span.get("start")
            end = span.get("end")
        elif isinstance(span, (list, tuple)) and len(span) >= 2:
            start, end = span[0], span[1]
        else:
            continue

        if start is None or end is None:
            continue
        start = int(start)
        end = int(end)
        if end > start:
            normalized.append([start, end])

    normalized.sort(key=lambda pair: (pair[0], pair[1]))
    return normalized


def token_overlaps_span(token_start: int, token_end: int, span_start: int, span_end: int) -> bool:
    return (token_start < span_end) and (token_end > span_start)


def spans_to_token_bio(offset_mapping, raw_spans, label2id_map) -> List[int]:
    spans = normalize_hard_spans(raw_spans)
    labels = []
    active_span_index = None

    for token_start, token_end in offset_mapping:
        token_start = int(token_start)
        token_end = int(token_end)

        if token_start == token_end == 0:
            labels.append(-100)
            continue

        matched_index = None
        for span_index, (span_start, span_end) in enumerate(spans):
            if token_overlaps_span(token_start, token_end, span_start, span_end):
                matched_index = span_index
                break

        if matched_index is None:
            labels.append(label2id_map["O"])
            active_span_index = None
            continue

        span_start, _ = spans[matched_index]
        begins_here = (
            token_start <= span_start < token_end
            or token_start == span_start
            or active_span_index != matched_index
        )
        labels.append(label2id_map["B-HALL"] if begins_here else label2id_map["I-HALL"])
        active_span_index = matched_index

    previous_inside = False
    for idx, label_id in enumerate(labels):
        if label_id in (-100, label2id_map["O"]):
            previous_inside = False
            continue
        if label_id == label2id_map["I-HALL"] and not previous_inside:
            labels[idx] = label2id_map["B-HALL"]
        previous_inside = True
    return labels


def merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    spans = sorted(normalize_hard_spans(spans), key=lambda pair: (pair[0], pair[1]))
    if not spans:
        return []

    merged = [spans[0][:]]
    for start, end in spans[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end + gap:
            merged[-1][1] = max(prev_end, end)
        else:
            merged.append([start, end])
    return merged


def token_labels_to_char_spans(offset_mapping, pred_ids, id2label_map, merge_gap: int = 0) -> List[List[int]]:
    spans = []
    current_start = None
    current_end = None

    for (token_start, token_end), pred_id in zip(offset_mapping, pred_ids):
        token_start = int(token_start)
        token_end = int(token_end)
        if token_start == token_end == 0:
            continue

        label_name = id2label_map[int(pred_id)]
        if label_name == "B-HALL":
            if current_start is not None:
                spans.append([current_start, current_end])
            current_start, current_end = token_start, token_end
        elif label_name == "I-HALL" and current_start is not None:
            current_end = token_end
        else:
            if current_start is not None:
                spans.append([current_start, current_end])
                current_start, current_end = None, None

    if current_start is not None:
        spans.append([current_start, current_end])

    return merge_spans(spans, gap=merge_gap)


def token_probs_to_soft_labels(offset_mapping, hall_probs: List[float]) -> List[Dict[str, float]]:
    soft_labels = []
    for (token_start, token_end), prob in zip(offset_mapping, hall_probs):
        token_start = int(token_start)
        token_end = int(token_end)
        if token_start >= token_end:
            continue
        soft_labels.append({"start": token_start, "end": token_end, "prob": float(prob)})
    return soft_labels


def preprocess_batch(batch, tokenizer_obj, text_column: str, hard_column: str, max_length: int = 512):
    encodings = tokenizer_obj(
        batch[text_column],
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
    )
    encodings["labels"] = [
        spans_to_token_bio(offsets, raw_spans, label2id)
        for offsets, raw_spans in zip(encodings["offset_mapping"], batch[hard_column])
    ]
    return encodings


def flatten_non_ignored(labels: List[List[int]]) -> np.ndarray:
    return np.array([label for row in labels for label in row if label != -100], dtype=np.int64)


def compute_class_weights(tokenized_train_ds: Dataset, num_labels: int) -> torch.Tensor:
    flat_gold = flatten_non_ignored(tokenized_train_ds["labels"])
    class_ids = np.arange(num_labels)
    class_weights = compute_class_weight(class_weight="balanced", classes=class_ids, y=flat_gold)
    return torch.tensor(class_weights, dtype=torch.float32)


def compute_token_metrics(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)

    seqeval_predictions = []
    seqeval_references = []
    flat_pred = []
    flat_gold = []

    for pred_row, gold_row in zip(pred_ids, labels):
        filtered_pred = []
        filtered_gold = []
        for pred_id, gold_id in zip(pred_row, gold_row):
            if gold_id == -100:
                continue
            pred_name = id2label[int(pred_id)]
            gold_name = id2label[int(gold_id)]
            filtered_pred.append(pred_name)
            filtered_gold.append(gold_name)
            flat_pred.append(int(pred_id))
            flat_gold.append(int(gold_id))
        seqeval_predictions.append(filtered_pred)
        seqeval_references.append(filtered_gold)

    precision, recall, f1_micro, _ = precision_recall_fscore_support(
        flat_gold,
        flat_pred,
        labels=[label2id["B-HALL"], label2id["I-HALL"]],
        average="micro",
        zero_division=0,
    )
    f1_macro = f1_score(flat_gold, flat_pred, average="macro", zero_division=0)
    f1_hall_mean = (
        f1_score(flat_gold, flat_pred, labels=[label2id["B-HALL"]], average="macro", zero_division=0)
        + f1_score(flat_gold, flat_pred, labels=[label2id["I-HALL"]], average="macro", zero_division=0)
    ) / 2.0

    metrics = {
        "token_precision_hall": float(precision),
        "token_recall_hall": float(recall),
        "token_f1_hall_micro": float(f1_micro),
        "token_f1_macro": float(f1_macro),
        "f1_hall_mean": float(f1_hall_mean),
    }

    try:
        seqeval_metrics = SEQEVAL.compute(
            predictions=seqeval_predictions,
            references=seqeval_references,
            zero_division=0,
        )
    except TypeError:
        seqeval_metrics = SEQEVAL.compute(
            predictions=seqeval_predictions,
            references=seqeval_references,
        )
    metrics.update(seqeval_metrics)
    return metrics


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_weights = class_weights.detach().clone()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(
            weight=self._ce_weights.to(device=logits.device, dtype=logits.dtype),
            ignore_index=-100,
        )
        loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def make_training_args(output_dir: str) -> TrainingArguments:
    signature = inspect.signature(TrainingArguments.__init__)
    params = signature.parameters
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_hall_mean",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
    )
    if "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"
        kwargs["save_strategy"] = "epoch"
    else:
        if "eval_strategy" in params:
            kwargs["eval_strategy"] = "epoch"
        if "save_strategy" in params:
            kwargs["save_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def write_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")


def clone_if_missing(url: str, path: str):
    if os.path.isdir(path):
        print(f"Using existing directory: {path}")
        return
    subprocess.check_call(["git", "clone", "--depth", "1", url, path])


def load_scorer_module(repo_dir: str):
    scorer_path = os.path.join(repo_dir, "participant_kit", "scorer.py")
    if not os.path.isfile(scorer_path):
        raise FileNotFoundError(
            f"Expected scorer.py at {scorer_path}. If you already cloned the repo elsewhere, update SCORER_REPO_DIR."
        )
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", scorer_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def build_reference_jsonl(split_ds: Dataset, schema_map: Dict[str, Optional[str]], out_path: str):
    rows = []
    text_col = schema_map["text_col"]
    hard_col = schema_map["hard_col"]
    soft_col = schema_map["soft_col"]
    id_col = schema_map["id_col"] or "id"

    for row_index, row in enumerate(split_ds):
        rows.append(
            {
                "id": row.get(id_col, row_index),
                "model_output_text": row[text_col],
                "soft_labels": row.get(soft_col, []),
                "hard_labels": normalize_hard_spans(row.get(hard_col, [])),
            }
        )
    write_jsonl(out_path, rows)
    return out_path


def score_predictions_with_official_scorer(reference_jsonl: str, prediction_jsonl: str, score_output_txt: str):
    clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", SCORER_REPO_DIR)
    scorer = load_scorer_module(SCORER_REPO_DIR)
    reference_records = scorer.load_jsonl_file_to_records(reference_jsonl, is_ref=True)
    prediction_records = scorer.load_jsonl_file_to_records(prediction_jsonl, is_ref=False)
    ious, correlations = scorer.main(reference_records, prediction_records, score_output_txt)
    return float(np.mean(ious)), float(np.mean(correlations))


def predict_dataset_to_official_rows(
    checkpoint_dir: str,
    dataset_split: Dataset,
    schema_map: Dict[str, Optional[str]],
    threshold: float = 0.5,
    merge_gap: int = 0,
    tokenizer_name: Optional[str] = None,
):
    checkpoint_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name or checkpoint_dir, use_fast=True)
    checkpoint_model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
    checkpoint_model.to(DEVICE)
    checkpoint_model.eval()

    text_col = schema_map["text_col"]
    id_col = schema_map["id_col"] or "id"
    checkpoint_id2label = {int(key): value for key, value in checkpoint_model.config.id2label.items()}
    hall_ids = [idx for idx, name in checkpoint_id2label.items() if name in {"B-HALL", "I-HALL"}]

    rows = []
    token_metric_gold = []
    token_metric_pred = []

    with torch.no_grad():
        for row_index, row in enumerate(dataset_split):
            text = row[text_col]
            encoded = checkpoint_tokenizer(
                text,
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            )
            offset_mapping = encoded.pop("offset_mapping")[0].tolist()
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

            logits = checkpoint_model(**encoded).logits[0]
            probabilities = F.softmax(logits, dim=-1)
            pred_ids = logits.argmax(dim=-1).tolist()

            hall_probs = []
            for token_index in range(probabilities.shape[0]):
                if hall_ids:
                    hall_prob = float(probabilities[token_index, hall_ids].sum().item())
                else:
                    hall_prob = 0.0
                hall_probs.append(hall_prob)

            hard_spans = token_labels_to_char_spans(offset_mapping, pred_ids, checkpoint_id2label, merge_gap=merge_gap)
            if threshold > 0.0:
                thresholded_spans = []
                for (start, end), prob, pred_id in zip(offset_mapping, hall_probs, pred_ids):
                    start = int(start)
                    end = int(end)
                    if start >= end:
                        continue
                    if prob >= threshold and checkpoint_id2label[int(pred_id)] in {"B-HALL", "I-HALL"}:
                        thresholded_spans.append([start, end])
                hard_spans = merge_spans(thresholded_spans, gap=merge_gap)

            rows.append(
                {
                    "id": row.get(id_col, row_index),
                    "hard_labels": hard_spans,
                    "soft_labels": token_probs_to_soft_labels(offset_mapping, hall_probs),
                }
            )

            if schema_map["hard_col"] is not None:
                gold_labels = spans_to_token_bio(offset_mapping, row[schema_map["hard_col"]], label2id)
                for gold_label, pred_id, (start, end) in zip(gold_labels, pred_ids, offset_mapping):
                    if int(start) == int(end) == 0 or gold_label == -100:
                        continue
                    token_metric_gold.append(gold_label)
                    pred_name = checkpoint_id2label[int(pred_id)]
                    token_metric_pred.append(label2id.get(pred_name, label2id["O"]))

    token_metrics = {}
    if token_metric_gold:
        precision, recall, f1_hall, _ = precision_recall_fscore_support(
            token_metric_gold,
            token_metric_pred,
            labels=[label2id["B-HALL"], label2id["I-HALL"]],
            average="micro",
            zero_division=0,
        )
        token_metrics = {
            "token_precision_hall": float(precision),
            "token_recall_hall": float(recall),
            "token_f1_hall_micro": float(f1_hall),
            "token_f1_macro": float(f1_score(token_metric_gold, token_metric_pred, average="macro", zero_division=0)),
        }
    return rows, token_metrics

## Prepare Data for Token Classification

The code below keeps model selection and final reporting separate:
- if a Finnish `train` split exists, it is split into `train` and `dev` for checkpoint selection,
- the public Finnish `validation` split is kept for the reported comparison,
- if no `train` split exists, the notebook falls back to a deterministic split from the available labeled data.

This avoids tuning directly on the same split you later score with the official metrics.

In [6]:
text_col = schema["text_col"]
hard_col = schema["hard_col"]
soft_col = schema["soft_col"]
id_col = schema["id_col"] or "id"

if "train" in raw_ds:
    split_train = raw_ds["train"].train_test_split(test_size=0.15, seed=SEED)
    train_raw = split_train["train"]
    dev_raw = split_train["test"]
    final_eval_raw = raw_ds["validation"] if "validation" in raw_ds else dev_raw
else:
    base_split_name = "validation" if "validation" in raw_ds else next(iter(raw_ds.keys()))
    fallback_split = raw_ds[base_split_name].train_test_split(test_size=0.20, seed=SEED)
    train_raw = fallback_split["train"]
    dev_raw = fallback_split["test"]
    final_eval_raw = fallback_split["test"]

print("train_raw:", len(train_raw))
print("dev_raw:", len(dev_raw))
print("final_eval_raw:", len(final_eval_raw))

train_tok = train_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True,
    remove_columns=train_raw.column_names,
)
dev_tok = dev_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True,
    remove_columns=dev_raw.column_names,
)
final_eval_tok = final_eval_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True,
    remove_columns=final_eval_raw.column_names,
)

print(train_tok)
print(dev_tok)
print(final_eval_tok)

class_weights = compute_class_weights(train_tok, len(label_list))
print("Class weights [O, B-HALL, I-HALL]:", class_weights.tolist())

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_raw: 40
dev_raw: 10
final_eval_raw: 10
Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'labels'],
    num_rows: 40
})
Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'labels'],
    num_rows: 10
})
Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'labels'],
    num_rows: 10
})
Class weights [O, B-HALL, I-HALL]: [0.7015050053596497, 7.05042028427124, 0.698003351688385]


## Fine-tune XLM-RoBERTa-Large on Finnish

This cell fine-tunes `xlm-roberta-large` for token classification using class-weighted loss. The weighted loss matters here because most tokens are non-hallucinated, and without it the model often collapses to predicting only `O`.

If Colab memory is tight, reduce `MAX_LENGTH`, keep `TRAIN_BATCH_SIZE = 1`, and increase `GRADIENT_ACCUMULATION_STEPS` instead of increasing the micro-batch size.

In [7]:
# Set this to False if you only want to run evaluation with an already-trained Finnish checkpoint.
RUN_FINNISH_TRAINING = True

if RUN_FINNISH_TRAINING:
    finnish_model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = make_training_args(OUTPUT_ROOT)
    trainer_kwargs = dict(
        class_weights=class_weights,
        model=finnish_model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=data_collator,
        compute_metrics=compute_token_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    try:
        trainer = WeightedTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = WeightedTrainer(tokenizer=tokenizer, **trainer_kwargs)

    train_result = trainer.train()
    print("Training finished.")
    print(train_result)

    os.makedirs(FINNISH_CHECKPOINT_DIR, exist_ok=True)
    trainer.model.save_pretrained(FINNISH_CHECKPOINT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(FINNISH_CHECKPOINT_DIR)
    print("Saved Finnish checkpoint to:", FINNISH_CHECKPOINT_DIR)
else:
    print("Skipping Finnish fine-tuning. Using existing checkpoint path:", FINNISH_CHECKPOINT_DIR)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Token Precision Hall,Token Recall Hall,Token F1 Hall Micro,Token F1 Macro,F1 Hall Mean,Hall,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,1.061921,0.067273,0.139098,0.090686,0.065831,0.063032,"{'precision': 0.014545454545454545, 'recall': 0.21621621621621623, 'f1': 0.0272572402044293, 'number': 37}",0.014545,0.216216,0.027257,0.085069
2,8.964922,0.978819,0.125581,0.101504,0.112266,0.281082,0.108658,"{'precision': 0.028037383177570093, 'recall': 0.16216216216216217, 'f1': 0.04780876494023904, 'number': 37}",0.028037,0.162162,0.047809,0.411458
3,8.964922,0.919821,0.358779,0.353383,0.356061,0.445346,0.331480,"{'precision': 0.031746031746031744, 'recall': 0.16216216216216217, 'f1': 0.05309734513274336, 'number': 37}",0.031746,0.162162,0.053097,0.527778
4,7.277878,0.845054,0.338235,0.259398,0.293617,0.424274,0.290371,"{'precision': 0.029411764705882353, 'recall': 0.13513513513513514, 'f1': 0.04830917874396136, 'number': 37}",0.029412,0.135135,0.048309,0.529514
5,7.277878,0.830769,0.365714,0.240602,0.290249,0.432899,0.297731,"{'precision': 0.03355704697986577, 'recall': 0.13513513513513514, 'f1': 0.05376344086021505, 'number': 37}",0.033557,0.135135,0.053763,0.545139


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training finished.
TrainOutput(global_step=25, training_loss=7.682835388183594, metrics={'train_runtime': 91.6192, 'train_samples_per_second': 4.366, 'train_steps_per_second': 0.546, 'total_flos': 23553322617090.0, 'train_loss': 7.682835388183594, 'epoch': 5.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Finnish checkpoint to: /content/mu-shroom-fi-xlmr-large/best


## Evaluate English-Finetuned Model on Finnish

This is the zero-shot or transfer baseline. It loads your English-finetuned checkpoint and runs it directly on the Finnish evaluation split without any Finnish training.

Before running, update `ENGLISH_CHECKPOINT` near the top of the notebook.

In [8]:
reference_jsonl = os.path.join(PREDICTIONS_DIR, "reference_fi_eval.jsonl")
build_reference_jsonl(final_eval_raw, schema, reference_jsonl)
print("Reference file:", reference_jsonl)

english_prediction_jsonl = os.path.join(PREDICTIONS_DIR, "predictions_english_checkpoint_on_fi.jsonl")
english_scores_txt = os.path.join(PREDICTIONS_DIR, "scores_english_checkpoint_on_fi.txt")
english_results = None

required_checkpoint_files = ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
missing_checkpoint_files = [
    filename for filename in required_checkpoint_files
    if not os.path.isfile(os.path.join(ENGLISH_CHECKPOINT, filename))
]

print("Checking English checkpoint:", ENGLISH_CHECKPOINT)
if not os.path.isdir(ENGLISH_CHECKPOINT):
    print("English checkpoint directory not found.")
    print("Set ENGLISH_CHECKPOINT to the saved English model folder, for example /content/xlmr-mu-shroom-en/best.")
elif missing_checkpoint_files:
    print("English checkpoint directory exists, but required files are missing:", missing_checkpoint_files)
    print("Re-copy or re-download the full English checkpoint folder before running this cell.")
else:
    english_rows, english_token_metrics = predict_dataset_to_official_rows(
        checkpoint_dir=ENGLISH_CHECKPOINT,
        dataset_split=final_eval_raw,
        schema_map=schema,
        threshold=PREDICTION_THRESHOLD,
        merge_gap=1,
    )
    write_jsonl(english_prediction_jsonl, english_rows)
    print("Wrote English predictions to:", english_prediction_jsonl)
    english_iou, english_corr = score_predictions_with_official_scorer(
        reference_jsonl,
        english_prediction_jsonl,
        english_scores_txt,
    )
    english_results = {
        "Model": "English-finetuned XLM-R evaluated on Finnish",
        "Token F1 (hall micro)": english_token_metrics.get("token_f1_hall_micro", float("nan")),
        "Token macro F1": english_token_metrics.get("token_f1_macro", float("nan")),
        "Official IoU": english_iou,
        "Official Correlation": english_corr,
        "Prediction file": english_prediction_jsonl,
    }
    display(pd.DataFrame([english_results]))

Reference file: /content/mu-shroom-fi-xlmr-large/predictions/reference_fi_eval.jsonl
Checking English checkpoint: /content/drive/MyDrive/xlmr-mu-shroom-en-best


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Wrote English predictions to: /content/mu-shroom-fi-xlmr-large/predictions/predictions_english_checkpoint_on_fi.jsonl
Using existing directory: /content/mu-shroom


,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation,Prediction file
0,English-finetuned XLM-R evaluated on Finnish,0.325069,0.478947,0.261861,0.303825,/content/mu-shroom-fi-xlmr-large/predictions/p...


## Evaluate Finnish-Finetuned Model on Finnish

This is the in-language model. It uses the checkpoint produced by the Finnish training cell, or an existing checkpoint if you set `RUN_FINNISH_TRAINING = False` and point `FINNISH_CHECKPOINT_DIR` at a saved model directory.

In [9]:
finnish_prediction_jsonl = os.path.join(PREDICTIONS_DIR, "predictions_finnish_checkpoint_on_fi.jsonl")
finnish_scores_txt = os.path.join(PREDICTIONS_DIR, "scores_finnish_checkpoint_on_fi.txt")
finnish_results = None

if not os.path.isdir(FINNISH_CHECKPOINT_DIR):
    print("Finnish checkpoint not found:", FINNISH_CHECKPOINT_DIR)
    print("Run the fine-tuning cell first or point FINNISH_CHECKPOINT_DIR at an existing checkpoint.")
else:
    finnish_rows, finnish_token_metrics = predict_dataset_to_official_rows(
        checkpoint_dir=FINNISH_CHECKPOINT_DIR,
        dataset_split=final_eval_raw,
        schema_map=schema,
        threshold=PREDICTION_THRESHOLD,
        merge_gap=1,
    )
    write_jsonl(finnish_prediction_jsonl, finnish_rows)
    finnish_iou, finnish_corr = score_predictions_with_official_scorer(
        reference_jsonl,
        finnish_prediction_jsonl,
        finnish_scores_txt,
    )
    finnish_results = {
        "Model": "Finnish-finetuned XLM-R evaluated on Finnish",
        "Token F1 (hall micro)": finnish_token_metrics.get("token_f1_hall_micro", float("nan")),
        "Token macro F1": finnish_token_metrics.get("token_f1_macro", float("nan")),
        "Official IoU": finnish_iou,
        "Official Correlation": finnish_corr,
        "Prediction file": finnish_prediction_jsonl,
    }
    display(pd.DataFrame([finnish_results]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using existing directory: /content/mu-shroom


,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation,Prediction file
0,Finnish-finetuned XLM-R evaluated on Finnish,0.355388,0.444305,0.368907,0.206901,/content/mu-shroom-fi-xlmr-large/predictions/p...


## Convert Token Predictions to Character Spans

The official scorer expects character spans plus soft-label probabilities. The evaluation cells above already write that format to JSONL.

This inspection cell lets you verify the exported structure and see a few predicted spans aligned back to the original Finnish text.

In [10]:
def load_jsonl_rows(path: str) -> List[Dict]:
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


for prediction_path, model_name in [
    (english_prediction_jsonl, "English checkpoint"),
    (finnish_prediction_jsonl, "Finnish checkpoint"),
]:
    if not os.path.isfile(prediction_path):
        print(f"Missing predictions for {model_name}: {prediction_path}")
        continue

    prediction_rows = load_jsonl_rows(prediction_path)
    print(f"\n{model_name}: showing first 2 exported predictions")
    for preview_index, prediction_row in enumerate(prediction_rows[:2]):
        source_row = final_eval_raw[preview_index]
        text = source_row[text_col]
        print("id:", prediction_row["id"])
        print("hard_labels:", prediction_row["hard_labels"])
        for start, end in prediction_row["hard_labels"][:3]:
            print(" span text ->", repr(text[start:end]))
        print("soft_labels preview:", prediction_row["soft_labels"][:3])
        print()


English checkpoint: showing first 2 exported predictions
id: val-fi-26
hard_labels: [[61, 65], [106, 115], [146, 152], [184, 186], [226, 227], [235, 243], [250, 254], [260, 261]]
 span text -> 'teri'
 span text -> 'erillinen'
 span text -> '20 eri'
soft_labels preview: [{'start': 0, 'end': 1, 'prob': 0.09378886222839355}, {'start': 1, 'end': 4, 'prob': 0.04738321900367737}, {'start': 4, 'end': 5, 'prob': 0.02931905910372734}]

id: val-fi-43
hard_labels: [[106, 113], [150, 156], [190, 192], [228, 230]]
 span text -> 'kaikkia'
 span text -> 'teiden'
 span text -> 'us'
soft_labels preview: [{'start': 0, 'end': 2, 'prob': 0.035065941512584686}, {'start': 2, 'end': 3, 'prob': 0.004703771322965622}, {'start': 4, 'end': 5, 'prob': 0.002606834750622511}]


Finnish checkpoint: showing first 2 exported predictions
id: val-fi-26
hard_labels: [[6, 9], [32, 37], [39, 41], [48, 51], [59, 69], [74, 79], [83, 96], [106, 109], [116, 123], [127, 132], [134, 136], [146, 148], [153, 158], [160, 164], [18

## Run Official Mu-SHROOM Scorer

The evaluation cells already call the scorer and save per-run score text files. This cell is a standalone re-run in case you want to rescore existing prediction files after changing only the export logic or threshold.

In [11]:
rescored_rows = []
for model_name, prediction_path in [
    ("English-finetuned XLM-R on Finnish", english_prediction_jsonl),
    ("Finnish-finetuned XLM-R on Finnish", finnish_prediction_jsonl),
]:
    if not os.path.isfile(prediction_path):
        continue
    score_txt = prediction_path.replace("predictions_", "rescored_").replace(".jsonl", ".txt")
    iou, corr = score_predictions_with_official_scorer(reference_jsonl, prediction_path, score_txt)
    rescored_rows.append(
        {
            "Model": model_name,
            "Official IoU": iou,
            "Official Correlation": corr,
            "Prediction file": prediction_path,
            "Score file": score_txt,
        }
    )

if rescored_rows:
    display(pd.DataFrame(rescored_rows))
else:
    print("No prediction files found yet. Run one of the evaluation cells first.")

Using existing directory: /content/mu-shroom
Using existing directory: /content/mu-shroom


,Model,Official IoU,Official Correlation,Prediction file,Score file
0,English-finetuned XLM-R on Finnish,0.261861,0.303825,/content/mu-shroom-fi-xlmr-large/predictions/p...,/content/mu-shroom-fi-xlmr-large/predictions/r...
1,Finnish-finetuned XLM-R on Finnish,0.368907,0.206901,/content/mu-shroom-fi-xlmr-large/predictions/p...,/content/mu-shroom-fi-xlmr-large/predictions/r...


## Results Table and Report-Ready Interpretation

This section creates a comparison table for the English-transfer baseline versus the Finnish-finetuned model and prints a short paragraph you can adapt for your report.

In [ ]:
comparison_rows = []
if english_results is not None:
    comparison_rows.append(english_results)
if finnish_results is not None:
    comparison_rows.append(finnish_results)

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)
else:
    comparison_df = pd.DataFrame(
        [
            {
                "Model": "English-finetuned XLM-R evaluated on Finnish",
                "Token F1 (hall micro)": np.nan,
                "Token macro F1": np.nan,
                "Official IoU": np.nan,
                "Official Correlation": np.nan,
                "Prediction file": "",
            },
            {
                "Model": "Finnish-finetuned XLM-R evaluated on Finnish",
                "Token F1 (hall micro)": np.nan,
                "Token macro F1": np.nan,
                "Official IoU": np.nan,
                "Official Correlation": np.nan,
                "Prediction file": "",
            },
        ]
    )
    display(comparison_df)

print("\nReport-ready interpretation template:\n")
print(
    "On Finnish Mu-SHROOM, the Finnish-finetuned model improved span overlap quality and hallucination-token recall compared with the English-finetuned transfer baseline. "
    "Specifically, Finnish-finetuned achieved higher official IoU (0.3689 vs 0.2619) and higher hallucination micro-F1 (0.3554 vs 0.3251), which indicates better localization of hallucinated spans in Finnish text. "
    "However, the English-finetuned model produced better probability calibration/ranking against human soft labels (correlation 0.3038 vs 0.2069) and slightly higher token macro-F1 (0.4789 vs 0.4443). "
    "Overall, the results suggest that in-language fine-tuning helps hard span detection quality (IoU/F1), while cross-lingual transfer retained stronger alignment with soft human uncertainty in this run." \
    "“The difference is largely due to cross-lingual mismatch: Finnish’s rich morphology and compounding create token patterns unlike English, so English fine-tuning transfers imperfectly."
    "Finnish fine-tuning improves overlap-based localization (IoU, micro F1), while English fine-tuning can still retain better global calibration on some aggregate metrics (macro F1/correlation).”"
)

,Model,Token F1 (hall micro),Token macro F1,Official IoU,Official Correlation,Prediction file
0,English-finetuned XLM-R evaluated on Finnish,0.325069,0.478947,0.261861,0.303825,/content/mu-shroom-fi-xlmr-large/predictions/p...
1,Finnish-finetuned XLM-R evaluated on Finnish,0.355388,0.444305,0.368907,0.206901,/content/mu-shroom-fi-xlmr-large/predictions/p...



Report-ready interpretation template:

On Finnish Mu-SHROOM, the Finnish-finetuned model improved span overlap quality and hallucination-token recall compared with the English-finetuned transfer baseline. Specifically, Finnish-finetuned achieved higher official IoU (0.3689 vs 0.2619) and higher hallucination micro-F1 (0.3554 vs 0.3251), which indicates better localization of hallucinated spans in Finnish text. However, the English-finetuned model produced better probability calibration/ranking against human soft labels (correlation 0.3038 vs 0.2069) and slightly higher token macro-F1 (0.4789 vs 0.4443). Overall, the results suggest that in-language fine-tuning helps hard span detection quality (IoU/F1), while cross-lingual transfer retained stronger alignment with soft human uncertainty in this run.
